# CTD Data Download — AML-6 (Cross-platform)

This notebook downloads raw cast files from the **AML-6 CTD** instrument to your computer over the instrument's built-in WiFi connection. It works on both **macOS** and **Windows** — set `PLATFORM` in the Configuration cell to match your operating system.

Run each cell in order using **Shift + Enter**. Before the numbered steps, run the **Setup** cell and edit the **Configuration** cell with the year, month (and optionally day), and your chosen folder name.

The notebook will then:

1. Create a local folder for the downloaded files
2. Connect to the CTD WiFi, download cast files matching the selected date pattern, and sort them into per-day subfolders
3. Remove any very small files that are not valid casts
4. Disconnect from the CTD WiFi

**Before you start:** make sure the CTD is powered on and within WiFi range.

> **Note:** when your computer joins the CTD network it may show a notification that the network has no internet connection. This is expected — the CTD is not connected to the internet.

> **Windows only:** this notebook requires the `paramiko` library. If it is not installed, run the pip install cell below the Setup cell, then restart the kernel.

## Setup

Run this cell first to load the required Python libraries and define the WiFi helper functions. You do not need to change anything here.

**Windows only:** this notebook requires the `paramiko` library for SFTP. If the import check below reports that it is missing, run the pip install cell and restart the kernel.

In [1]:
# Windows only — uncomment and run if paramiko is not installed, then restart the kernel.
# !pip install paramiko

In [2]:
import os
import re
import tempfile
import subprocess
import time
from collections import defaultdict
from pathlib import Path

try:
    import paramiko
    _paramiko_ok = True
except ImportError:
    _paramiko_ok = False


# ── macOS WiFi helpers ───────────────────────────────────────────
def _mac_get_iface():
    result = subprocess.run(
        ['networksetup', '-listallhardwareports'],
        capture_output=True, text=True
    )
    lines = result.stdout.splitlines()
    for i, line in enumerate(lines):
        if 'Wi-Fi' in line or 'AirPort' in line:
            for next_line in lines[i + 1: i + 4]:
                if next_line.startswith('Device:'):
                    return next_line.split('Device:')[1].strip()
    return 'en0'


def _mac_connect(ssid, password):
    iface = _mac_get_iface()
    subprocess.run(
        ['networksetup', '-setairportnetwork', iface, ssid, password],
        capture_output=True
    )
    time.sleep(5)


def _mac_disconnect(ssid):
    iface = _mac_get_iface()
    subprocess.run(
        ['networksetup', '-removepreferredwirelessnetwork', iface, ssid],
        capture_output=True
    )


def _mac_sftp_download(host, port, user, password, remote_dir, date_pattern, local_dir):
    """Download matching files via /usr/bin/expect + sftp.

    Used on macOS because the Local Network privacy sandbox blocks Python's
    socket module (and therefore paramiko) from reaching 172.18.x.x addresses.
    System binaries such as /usr/bin/sftp are exempt from this restriction.

    Host-key checking is disabled so that CTD firmware updates or hardware
    replacements (which change the instrument's host key) never block a download.
    """
    script = (
        '#!/usr/bin/expect -f\n'
        'set timeout 60\n'
        f'spawn /usr/bin/sftp'
        f' -o StrictHostKeyChecking=no'
        f' -o UserKnownHostsFile=/dev/null'
        f' -P {port} {user}@{host}\n'
        'expect "password:"\n'
        f'send "{password}\\r"\n'
        'expect "sftp>"\n'
        f'send "cd {remote_dir}\\r"\n'
        'expect "sftp>"\n'
        f'send "lcd \\"{local_dir}\\"\\r"\n'
        'expect "sftp>"\n'
        f'send "mget *{date_pattern}*\\r"\n'
        'expect -timeout 120 "sftp>"\n'
        'send "bye\\r"\n'
        'expect eof\n'
    )
    with tempfile.NamedTemporaryFile(mode='w', suffix='.exp', delete=False) as fh:
        fh.write(script)
        script_path = fh.name
    try:
        result = subprocess.run(
            ['/usr/bin/expect', script_path],
            capture_output=True, text=True, timeout=180
        )
        return result.stdout, result.stderr
    finally:
        os.unlink(script_path)


# ── Windows WiFi helpers ─────────────────────────────────────────
def _win_connect(ssid, password):
    lines = [
        '<?xml version="1.0"?>',
        '<WLANProfile xmlns="http://www.microsoft.com/networking/WLAN/profile/v1">',
        f'  <name>{ssid}</name>',
        f'  <SSIDConfig><SSID><name>{ssid}</name></SSID></SSIDConfig>',
        '  <connectionType>ESS</connectionType>',
        '  <connectionMode>manual</connectionMode>',
        '  <MSM><security>',
        '    <authEncryption>',
        '      <authentication>WPA2PSK</authentication>',
        '      <encryption>AES</encryption>',
        '      <useOneX>false</useOneX>',
        '    </authEncryption>',
        '    <sharedKey>',
        '      <keyType>passPhrase</keyType>',
        '      <protected>false</protected>',
        f'      <keyMaterial>{password}</keyMaterial>',
        '    </sharedKey>',
        '  </security></MSM>',
        '</WLANProfile>',
    ]
    profile_xml = '\n'.join(lines)
    xml_file = tempfile.NamedTemporaryFile(
        mode='w', suffix='.xml', delete=False, prefix='ctd_wifi_'
    )
    xml_file.write(profile_xml)
    xml_file.close()
    try:
        subprocess.run(
            ['netsh', 'wlan', 'add', 'profile', f'filename={xml_file.name}', 'user=current'],
            capture_output=True
        )
    finally:
        os.unlink(xml_file.name)
    subprocess.run(
        ['netsh', 'wlan', 'connect', f'name={ssid}'],
        capture_output=True
    )
    time.sleep(5)


def _win_disconnect(ssid):
    subprocess.run(
        ['netsh', 'wlan', 'delete', 'profile', f'name={ssid}'],
        capture_output=True
    )
    subprocess.run(['netsh', 'wlan', 'disconnect'], capture_output=True)


if not _paramiko_ok:
    print('Ready. (paramiko not found — Windows SFTP will not work; run the pip install cell above if on Windows.)')
else:
    print('Ready.')

Ready.


## Configuration

**Edit the settings below, then continue running cells from top to bottom.**

| Setting | Description |
|---|---|
| `PLATFORM` | `'mac'` for macOS or `'windows'` for Windows |
| `SAVE_DIR` | Parent folder on your computer where the new subfolder will be created |
| `FOLDER_NAME` | Name for the new subfolder (spaces are fine) |
| `YEAR` | Year of the casts to download |
| `MONTH` | Month of the casts to download (1–12) |
| `DAY` | Day of the casts to download (1–31), or `None` to download the whole month |
| `MIN_FILE_SIZE_KB` | Files smaller than this (in kB) are deleted after download — these are usually aborted casts that did not reach depth |
| `CTD_SSID` | WiFi network name broadcast by your CTD (shown on the instrument label or in the AML instrument manager) |
| `CTD_PASSWORD` | WiFi password for the CTD network |
| `CTD_HOST` | IP address of the CTD when connected to its WiFi (shown in the AML instrument manager) |
| `CTD_PORT` | SFTP port — 22 for all current AML instruments |
| `CTD_USER` | SFTP login username for the CTD |
| `CTD_PASS` | SFTP login password for the CTD |

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────
# Operating system: 'mac' for macOS, 'windows' for Windows.
PLATFORM = 'mac'

# Parent directory where the new folder will be created.
SAVE_DIR = Path.home() / 'Documents'

# Name for the new folder (spaces are fine).
FOLDER_NAME = 'CTD Data'

# Year and month of the casts to download.
YEAR  = 2026
MONTH = 5      # 1–12
DAY   = None   # Set to a day (1–31) to download a specific day, or None for the whole month

# Files smaller than this (in kB) will be deleted after download.
MIN_FILE_SIZE_KB = 50

# ── CTD NETWORK CREDENTIALS ─────────────────────────────────────
# Update these values to match your instrument. The credentials below
# are for the UBC/BOGL AML-6 (serial A60178) and are shown as examples.
# Your instrument's WiFi name and IP address are printed on its label
# and visible in the AML Instrument Manager software.

# WiFi network name (SSID) broadcast by the CTD.
# Example: 'AML_A60178'
CTD_SSID     = 'AML_A60178'

# WiFi password for the CTD network.
# Example: 'A60178A60178'
CTD_PASSWORD = 'A60178A60178'

# IP address of the CTD when connected to its WiFi.
# Example: '172.18.127.2'
CTD_HOST     = '172.18.127.2'

# SFTP port — 22 for all current AML instruments.
CTD_PORT     = 22

# SFTP login credentials — the same for all AML instruments.
CTD_USER     = 'aml'
CTD_PASS     = 'oceanographic'

# Directory on the CTD where cast files are stored — do not change.
CTD_LOG_DIR  = 'log'
# ────────────────────────────────────────────────────────────────

MIN_FILE_SIZE = MIN_FILE_SIZE_KB * 1024
MONTH_STR     = f'{YEAR}-{MONTH:02d}'
DATE_PATTERN  = MONTH_STR if DAY is None else f'{MONTH_STR}-{DAY:02d}'
BASE_DIR      = SAVE_DIR / FOLDER_NAME

print(f'Platform       : {PLATFORM}')
print(f'Base directory : {BASE_DIR}')
print(f'Date pattern   : {DATE_PATTERN}')

## Step 1 — Create Output Directory

Creates the base folder `SAVE_DIR / FOLDER_NAME` if it does not already exist. Files for each day will be placed in dated subfolders inside this directory (e.g. `CTD Data/2026-05-04/`).

In [4]:
BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Directory ready: {BASE_DIR}')

Directory ready: /Users/hjb62/Documents/CTD Data


## Step 2 — Download Files from the CTD

Connects to the CTD WiFi and downloads every file in the instrument's `log` directory whose name matches the selected date pattern (the full month when `DAY` is `None`, or a specific day when `DAY` is set). Files are then sorted into per-day subfolders inside the base directory.

- **macOS:** WiFi is managed automatically via `networksetup`.
- **Windows:** WiFi is managed automatically via `netsh`. If automatic connection fails (this can happen if your user account lacks permission to add WiFi profiles), a message will appear — connect manually via the taskbar and re-run this cell.

If no files match, the cell will say so — update `YEAR`, `MONTH` (and `DAY` if set) in the configuration cell and re-run.

In [5]:
print(f'Connecting to {CTD_SSID} ...', end=' ', flush=True)
if PLATFORM == 'mac':
    _mac_connect(CTD_SSID, CTD_PASSWORD)
else:
    _win_connect(CTD_SSID, CTD_PASSWORD)
print('done.')

print(f'Downloading files matching "{DATE_PATTERN}" from {CTD_HOST} ...')

if PLATFORM == 'mac':
    _n_before = len(list(BASE_DIR.glob(f'*{DATE_PATTERN}*')))
    _stdout, _stderr = _mac_sftp_download(
        CTD_HOST, CTD_PORT, CTD_USER, CTD_PASS,
        CTD_LOG_DIR, DATE_PATTERN, BASE_DIR
    )
    if _stdout:
        print(_stdout)
    if _stderr:
        print(_stderr)
    _n_after = len(list(BASE_DIR.glob(f'*{DATE_PATTERN}*')))
    _n_downloaded = _n_after - _n_before
    if _n_downloaded > 0:
        print(f'Downloaded {_n_downloaded} file(s).')
    else:
        print(f'No new files matching "{DATE_PATTERN}" were downloaded.')
        print('Check that the CTD is powered on and within WiFi range, then try again.')

else:  # Windows — paramiko SFTP
    _n_downloaded = 0
    try:
        _transport = paramiko.Transport((CTD_HOST, CTD_PORT))
        _transport.connect(username=CTD_USER, password=CTD_PASS)
        _sftp = paramiko.SFTPClient.from_transport(_transport)

        _sftp.chdir(CTD_LOG_DIR)
        _matching = sorted(f for f in _sftp.listdir() if DATE_PATTERN in f)

        if not _matching:
            print(f'No files matching "{DATE_PATTERN}" were found on the CTD.')
            print('Update YEAR, MONTH (and DAY if set) in the configuration cell and try again.')
        else:
            for _fname in _matching:
                _local = BASE_DIR / _fname
                print(f'  {_fname} ...', end=' ', flush=True)
                _sftp.get(_fname, str(_local))
                print(f'{_local.stat().st_size / 1024:.1f} kB')
                _n_downloaded += 1
            print(f'\nDownloaded {_n_downloaded} file(s).')

        _sftp.close()
        _transport.close()

    except Exception as _e:
        print(f'\nConnection failed: {_e}')
        print('If the WiFi connected but SFTP failed, check the CTD is powered on and in range.')
        print('If the WiFi did not connect, join the CTD network manually from the taskbar and re-run.')

# Sort downloaded files into per-day subdirectories
_flat = [f for f in BASE_DIR.glob(f'*{DATE_PATTERN}*') if f.is_file()]
if _flat:
    _by_day = defaultdict(list)
    for _f in _flat:
        _m = re.search(r'(\d{4}-\d{2}-\d{2})', _f.name)
        if _m:
            _by_day[_m.group(1)].append(_f)
    for _day, _files in sorted(_by_day.items()):
        _day_dir = BASE_DIR / _day
        _day_dir.mkdir(exist_ok=True)
        for _f in sorted(_files):
            _f.rename(_day_dir / _f.name)
    print(f'\nFiles sorted into {len(_by_day)} day folder(s):')
    for _day in sorted(_by_day):
        print(f'  {_day}/')
    print('\nDownload complete.')

Connecting to AML_A60178 ... done.
spawn /usr/bin/sftp -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -P 22 aml@172.18.127.2

** WARNING: connection is not using a post-quantum key exchange algorithm.

** This session may be vulnerable to "store now, decrypt later" attacks.

** The server may need to be upgraded. See https://openssh.com/pq.html


aml@172.18.127.2's password: 
Connected to 172.18.127.2.
sftp> cd log
sftp> lcd "/Users/hjb62/Documents/CTD Data"
sftp> mget *2026-05*
Fetching /media/sd/log/aml_log_2026-05-04_10-03-19.aml to aml_log_2026-05-04_10-03-19.aml

aml_log_2026-05-04_10-03-19.aml                 0%    0     0.0KB/s   --:-- ETA
aml_log_2026-05-04_10-03-19.aml               100%  709KB   1.5MB/s   00:00    
Fetching /media/sd/log/aml_log_2026-05-04_10-11-14.aml to aml_log_2026-05-04_10-11-14.aml

aml_log_2026-05-04_10-11-14.aml                 0%    0     0.0KB/s   --:-- ETA
aml_log_2026-05-04_10-11-14.aml                75% 2208KB   2.2MB/s   00:00 ETA
a

## Step 3 — Remove Small Files

Each time the CTD enters the water it creates a new log file, even if it never reaches depth. These aborted-cast files are typically only a few kB. The cell below deletes any downloaded file smaller than `MIN_FILE_SIZE_KB` kB and reports what was kept.

In [6]:
_day_dirs = sorted(
    d for d in BASE_DIR.iterdir()
    if d.is_dir() and d.name.startswith(MONTH_STR)
)

if not _day_dirs:
    print(f'No day folders found for {MONTH_STR} — run Steps 1 and 2 first.')
else:
    for _day_dir in _day_dirs:
        _all   = [f for f in _day_dir.iterdir() if f.is_file()]
        _small = sorted(f for f in _all if f.stat().st_size < MIN_FILE_SIZE)
        _kept  = sorted(f for f in _all if f.stat().st_size >= MIN_FILE_SIZE)
        if not _all:
            continue
        print(f'\n{_day_dir.name}:')
        if _small:
            for _f in _small:
                print(f'  Removing  {_f.name}  ({_f.stat().st_size / 1024:.1f} kB)')
                _f.unlink()
        else:
            print(f'  No files smaller than {MIN_FILE_SIZE_KB} kB.')
        for _f in _kept:
            print(f'  Kept      {_f.name}  ({_f.stat().st_size / 1024:.1f} kB)')


2026-05-04:
  Removing  aml_log_2026-05-04_11-13-45.aml  (4.2 kB)
  Removing  aml_log_2026-05-04_11-24-43.aml  (5.6 kB)
  Removing  aml_log_2026-05-04_11-34-34.aml  (4.6 kB)
  Removing  aml_log_2026-05-04_11-45-26.aml  (5.5 kB)
  Kept      2026-05-04_extracted.csv  (10495.6 kB)
  Kept      aml_log_2026-05-04_10-03-19.aml  (709.0 kB)
  Kept      aml_log_2026-05-04_10-11-14.aml  (2915.2 kB)
  Kept      aml_log_2026-05-04_10-41-42.aml  (2570.3 kB)
  Kept      aml_log_2026-05-04_11-02-55.aml  (1479.9 kB)
  Kept      aml_log_2026-05-04_11-14-35.aml  (1382.4 kB)
  Kept      aml_log_2026-05-04_11-25-38.aml  (1219.9 kB)
  Kept      aml_log_2026-05-04_11-35-37.aml  (1338.5 kB)
  Kept      aml_log_2026-05-04_11-45-36.aml  (281.1 kB)

2026-05-05:
  Removing  aml_log_2026-05-05_11-12-10.aml  (4.9 kB)
  Kept      2026-05-05_extracted.csv  (7558.7 kB)
  Kept      aml_log_2026-05-05_09-47-52.aml  (100.7 kB)
  Kept      aml_log_2026-05-05_09-52-16.aml  (930.6 kB)
  Kept      aml_log_2026-05-05_09-59-

## Step 4 — Disconnect from the CTD WiFi

Removes the CTD network from your preferred networks and disconnects. You will need to reconnect to your usual network manually afterwards.

In [ ]:
if PLATFORM == 'mac':
    _mac_disconnect(CTD_SSID)
    print(f'Removed {CTD_SSID} from preferred WiFi networks.')
else:
    _win_disconnect(CTD_SSID)
    print(f'Disconnected from {CTD_SSID} and removed its profile.')
print()
print('To reconnect to the internet:')
print('  Click the WiFi icon in the menu bar / taskbar and select your usual network.')

---
## Done

Your CTD files are saved in dated subfolders inside `BASE_DIR`. Open **CTD_Data_Extraction.ipynb** or **CTD_Data_Extraction_mmap.ipynb** and set `DATE_PATTERNS` to the relevant date to process the casts.